In [5]:
import os

BASE_DIR = "hf"

os.environ["HF_HOME"] = f"{BASE_DIR}/home"
os.environ["HF_DATASETS_CACHE"] = f"{BASE_DIR}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{BASE_DIR}/transformers"
os.environ["HF_METRICS_CACHE"] = f"{BASE_DIR}/metrics"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [3]:
import pandas as pd

df = pd.read_csv("datasets/hindi_english_parallel.csv")
df.dropna(inplace=True)
df["english"] = df["english"].str.strip()
df["hindi"] = df["hindi"].str.strip()

df = df[
    (df["english"].str.len() < 200) &
    (df["hindi"].str.len() < 200)
]

df = df.sample(n=60000, random_state=42).reset_index(drop=True)

print("Using rows:", len(df))

Using rows: 60000


In [35]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

In [25]:
from transformers import MarianTokenizer
from torch.utils.data import Dataset

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

class MarianDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.src = df["english"].tolist()
        self.tgt = df["hindi"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.src[idx],
            max_length=self.max_len,
            truncation=True,
            padding="max_length"
        )

        labels = self.tokenizer(
            text_target=self.tgt[idx],
            max_length=self.max_len,
            truncation=True,
            padding="max_length"
        )["input_ids"]

        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in labels
        ]

        inputs["labels"] = labels
        return inputs


/workspace/exp_armaan/env/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [27]:
teacher = MarianMTModel.from_pretrained(MODEL_NAME).cuda() 

args = Seq2SeqTrainingArguments(
    output_dir="./teacher",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    fp16=True,
    learning_rate=2e-5,
    num_train_epochs=3,

    eval_strategy="steps",
    eval_steps=5000,
    save_steps=5000,
    logging_steps=100,

    report_to="none",
    load_best_model_at_end=True
) 

trainer = Seq2SeqTrainer(
    model=teacher,
    args=args,
    train_dataset=MarianDataset(train_df, tokenizer),
    eval_dataset=MarianDataset(val_df, tokenizer),
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipykernel_260183/91565656.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
5000,2.553000,2.717858


/workspace/exp_armaan/env/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=5064, training_loss=2.8289965619031476, metrics={'train_runtime': 350.1461, 'train_samples_per_second': 462.664, 'train_steps_per_second': 14.463, 'total_flos': 5491535118336000.0, 'train_loss': 2.8289965619031476, 'epoch': 3.0})

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_DIR = "teacher/checkpoint-5064"   
model = MarianMTModel.from_pretrained(MODEL_DIR).cuda()
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

model.eval()

In [30]:
import torch
from tqdm import tqdm

def generate_translations(texts, batch_size=16, max_len=128):
    preds = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=max_len,
                num_beams=4
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend(decoded)

    return preds

In [31]:
predictions = generate_translations(val_df["english"].tolist())
references = val_df["hindi"].tolist()


00%|██████████| 375/375 [00:57<00:00,  6.52it/s]

In [32]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

print(f"BLEU score: {bleu.score:.2f}")

BLEU score: 27.58


In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
import sacrebleu

DEVICE = "cuda"

BASE_MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"

FINETUNED_MODEL_DIR = "teacher/checkpoint-5064"

tokenizer = MarianTokenizer.from_pretrained(BASE_MODEL_NAME)

# Load models
base_model = MarianMTModel.from_pretrained(BASE_MODEL_NAME).to(DEVICE)
finetuned_model = MarianMTModel.from_pretrained(FINETUNED_MODEL_DIR).to(DEVICE)

base_model.eval()
finetuned_model.eval()

In [37]:
def generate_translations(model, texts, batch_size=16, max_len=128):
    preds = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=max_len,
                num_beams=4
            )

        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )
        preds.extend(decoded)

    return preds

In [ ]:
sources = val_df["english"].tolist()
references = val_df["hindi"].tolist()

print("Generating with BASE (non-fine-tuned) model")
base_preds = generate_translations(base_model, sources)

print("Generating with FINE-TUNED model")
finetuned_preds = generate_translations(finetuned_model, sources)

In [ ]:
base_bleu = sacrebleu.corpus_bleu(
    base_preds,
    [references]
)

finetuned_bleu = sacrebleu.corpus_bleu(
    finetuned_preds,
    [references]
)

print("\nBLEU SCORE COMPARISON")
print(f"Base model BLEU       : {base_bleu.score:.2f}")
print(f"Fine-tuned model BLEU : {finetuned_bleu.score:.2f}")


BLEU SCORE COMPARISON
Base model BLEU : 23.66
Fine-tuned model BLEU : 27.58
